# Johanan's Convex Optimizer 

In [ ]:
def gradient(weights, monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt):
    grad = np.zeros_like(weights)
    for t in range(len(monthly_data)):
        stock_returns = monthly_data.iloc[t, :].values
        ff_factors = ff3_monthly.iloc[t, :].values
        portfolio_return = np.dot(weights, stock_returns - ff_factors[3])
        expected_return = mkt_opt * ff_factors[0] + smb_opt * ff_factors[1] + hml_opt * ff_factors[2]
        grad += 2 * (portfolio_return - expected_return) * (stock_returns - ff_factors[3])  # Gradient for stock i
    return grad


In [ ]:
def svrg_optimizer2(monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt, base_weights, max_iter=100, batch_size=10, learning_rate=0.01):
    # Initialize weights
    weights = np.copy(base_weights)
    losslist=[]
    weightlist=[]
    
    # Full gradient at snapshot weights (w^k)
    full_grad = gradient(weights, monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt)
    
    for iteration in range(max_iter):
        # Randomly select a minibatch of data
        batch_indices = np.random.choice(len(monthly_data), batch_size, replace=False)
        batch_data = monthly_data.iloc[batch_indices, :]
        batch_ff3 = ff3_monthly.iloc[batch_indices, :]
        
        # Compute the gradient for the minibatch
        minibatch_grad = np.zeros_like(weights)
        for t in range(len(batch_data)):
            stock_returns = batch_data.iloc[t, :].values
            ff_factors = batch_ff3.iloc[t, :].values
            portfolio_return = np.dot(weights, stock_returns - ff_factors[3])
            expected_return = mkt_opt * ff_factors[0] + smb_opt * ff_factors[1] + hml_opt * ff_factors[2]
            minibatch_grad += 2 * (portfolio_return - expected_return) * (stock_returns - ff_factors[3])
        
        # Variance-reduced gradient update rule
        weights -= learning_rate * (minibatch_grad - full_grad + gradient(weights, monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt))
        weightlist.append(weights)
        loss = error_term(weights, monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt)
        losslist.append(loss)
        # Print the loss (optional)
        if iteration % 10 == 0:
            print(f"Iteration {iteration}, Loss: {loss}")
    
    return losslist, weights,weightlist


In [ ]:
def svrg_optimizer():
    global weights
    prev_loss = float('inf')
    losslist=[]

    for iteration in range(max_iter):
        # Compute the full gradient at current weights
        full_grad = gradient(weights, monthly_data.iloc[:,:-1], ff3_monthly, mkt_opt, smb_opt, hml_opt)

        # Minibatch selection
        batch_indices = np.random.choice(len(monthly_data), batch_size, replace=False)
        batch_data = monthly_data.iloc[batch_indices, :]
        batch_data=batch_data.iloc[:,:-1]
        batch_ff3 = ff3_monthly.iloc[batch_indices, :]

        # Compute gradient on minibatch
        minibatch_grad = np.zeros_like(weights)
        for t in range(len(batch_data)):
            stock_returns = batch_data.iloc[t, :].values
            ff_factors = batch_ff3.iloc[t, :].values
            portfolio_return = np.dot(weights, stock_returns - ff_factors[3])
            expected_return = mkt_opt * ff_factors[0] + smb_opt * ff_factors[1] + hml_opt * ff_factors[2]
            minibatch_grad += 2 * (portfolio_return - expected_return) * (stock_returns - ff_factors[3])  # Gradient for stock i

        # Combine full and minibatch gradients (SVRG update)
        gradient_update = minibatch_grad - full_grad + full_grad
        weights -= learning_rate * gradient_update

        # Normalize to satisfy weight sum constraint
        weights = weights / np.sum(weights)
        # print('t0\n', weights)
        # print('t0\n',sum(weights))
        # print('t0\n',max_stocks_in_portfolio)

        # Apply weight change penalty to respect the base weights (absolute change)
        # weight_changes = np.abs(weights - base_weights)
        # penalty = np.sum(weight_changes)
        # if penalty > 1:  # If penalty is too large, we can adjust learning rate or add penalty
        #     weights -= learning_rate * penalty
        # print('t1\n', weights)
        # print('t1\n',sum(weights))

        # Apply transaction cost constraint (penalty method)
        transaction_cost = 0
        for i in range(num_assets):
            ticker = tickers[i]
            transaction_cost += np.abs(weights[i] - base_weights[i]) * transaction_costs[ticker] / price_per_share[ticker]
        if transaction_cost > transaction_cost_limit:
            # Penalize the weights for exceeding the transaction cost limit
            weights -= learning_rate * (transaction_cost - transaction_cost_limit)
        # print('t2\n', weights)
        # print('t2\n',sum(weights))
        
            

        # Ensure non-negative weights (ReLU)
        weights = np.maximum(0, weights)
        # print('t3\n', weights)
        # print('t3\n',sum(weights))
        # print('t3\n', max_stocks_in_portfolio)
        # Limit the number of stocks in the portfolio
        if np.sum(weights > 0) > max_stocks_in_portfolio:
            # Zero out the smallest weights to limit the number of stocks
            sorted_indices = np.argsort(weights)
            weights[sorted_indices[:len(weights) - max_stocks_in_portfolio]] = 0
        
        weights = weights / np.sum(weights)
        # print('t4\n', weights)
        # print('t4\n',sum(weights))

        # Check if the error is converging
        current_loss = error_term(weights, monthly_data.iloc[:,:-1], ff3_monthly, mkt_opt, smb_opt, hml_opt)
        losslist.append(current_loss)
        # if current_loss < 0.014:
        #     print(f"Converged at iteration {iteration} with loss {current_loss}")
        #     break
        # prev_loss = current_loss

    return losslist,weights



In [ ]:
mkt_opt, smb_opt, hml_opt = 1.0, 0, 0
learning_rate = 0.01
max_iter = 10000
batch_size = 32
tolerance = 1e-3
transaction_cost_limit = 2000  
max_stocks_in_portfolio = 50
num_assets = len(tickers)
weights = np.random.rand(num_assets)  # Random initialization of weights
weights /= np.sum(weights)
transaction_costs = t_cost
price_per_share=s_price
hahla3,hjaL3=svrg_optimizer()
port_weights=pd.DataFrame(hjaL3, index=tickers, columns=['weights'])
# port_weights['weights'] = port_weights['weights'].round(2)
port_weights
ploterror(hahla3)
portfolio_betas(port_weights)



In [ ]:
def lamb_optimizer(weights, gradient, learning_rate, beta1, beta2, epsilon, trust_ratio):
    m = np.zeros_like(weights)
    v = np.zeros_like(weights)
    
    for t in range(num_iterations):
        # Compute gradient
        gradient = compute_gradient(weights, monthly_data, ff3_monthly, mkt_opt, smb_opt, hml_opt)
        
        # Update first moment (m)
        m = beta1 * m + (1 - beta1) * gradient
        
        # Update second moment (v)
        v = beta2 * v + (1 - beta2) * gradient ** 2
        
        # Compute trust ratio
        trust_ratio = np.linalg.norm(m) / np.linalg.norm(gradient)
        
        # Update learning rate
        learning_rate = learning_rate * trust_ratio
        
        # Update weights
        weights = weights - learning_rate * m / (np.sqrt(v) + epsilon)
    
    return weights